In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
INPUT_PATH = r"D:\Final_input_2.xlsx" 
OUTPUT_PATH = r"D:\Champion Club RFM Data_2.xlsx"
df = pd.read_excel(INPUT_PATH)

In [3]:
df.head()

,MASON.SAPNEY MASON CONTACT NUMBER,PURCHASE INTENT REQUEST DATE.FULLDATE,PURCHASE INTENT TSE TOTAL BAGS,Mobile Number,D SAPNEY LOCATION.SAPNEY REGION,D SAPNEY LOCATION.SAPNEY BRANCH,D SAPNEY LOCATION.SAPNEY TERRITORY
0,38352-9165908704,2026-02-18,80,9165908704,STNA,SINGRAULI,SINGRAULI
1,38029-6265452227,2025-05-16,80,6265452227,STNA,SINGRAULI,SINGRAULI
2,38179-8225974840,2025-07-31,80,8225974840,STNA,SINGRAULI,SINGRAULI
3,36007-9770858175,2025-12-06,80,9770858175,STNA,SINGRAULI,SINGRAULI
4,38222-9685840059,2025-07-31,80,9685840059,STNA,SINGRAULI,SINGRAULI


In [4]:
# Define columns explicitly
date_col = 'PURCHASE INTENT REQUEST DATE.FULLDATE'
qty_col = 'PURCHASE INTENT TSE TOTAL BAGS'
mobile_col = 'Mobile Number'
region_col = 'D SAPNEY LOCATION.SAPNEY REGION'
branch_col = 'D SAPNEY LOCATION.SAPNEY BRANCH'
territory_col = 'D SAPNEY LOCATION.SAPNEY TERRITORY'

In [5]:
# 2. Data Cleaning & Prep
# ==============================
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
df = df.dropna(subset=[date_col, mobile_col])

In [6]:
df.head()

,MASON.SAPNEY MASON CONTACT NUMBER,PURCHASE INTENT REQUEST DATE.FULLDATE,PURCHASE INTENT TSE TOTAL BAGS,Mobile Number,D SAPNEY LOCATION.SAPNEY REGION,D SAPNEY LOCATION.SAPNEY BRANCH,D SAPNEY LOCATION.SAPNEY TERRITORY
0,38352-9165908704,2026-02-18,80,9165908704,STNA,SINGRAULI,SINGRAULI
1,38029-6265452227,2025-05-16,80,6265452227,STNA,SINGRAULI,SINGRAULI
2,38179-8225974840,2025-07-31,80,8225974840,STNA,SINGRAULI,SINGRAULI
3,36007-9770858175,2025-12-06,80,9770858175,STNA,SINGRAULI,SINGRAULI
4,38222-9685840059,2025-07-31,80,9685840059,STNA,SINGRAULI,SINGRAULI


In [7]:
#Analysis date = day after last transaction
analysis_date = df[date_col].max() + pd.Timedelta(days=1)

In [8]:
analysis_date

Timestamp('2026-04-01 00:00:00')

In [9]:
# Create Purchase Month
df['purchase_month'] = df[date_col].dt.to_period('M')

In [10]:
df['purchase_month'] = df[date_col].dt.to_period('M').astype(str) + '-M'

In [11]:
df['purchase_month'].head()

0    2026-02-M
1    2025-05-M
2    2025-07-M
3    2025-12-M
4    2025-07-M
Name: purchase_month, dtype: object

In [12]:
# 5. CUSTOMER ELIGIBILITY (Monthly ≥50)
# ==============================
monthly_sum = (
    df.groupby([mobile_col, 'purchase_month'])[qty_col]
      .sum()
      .reset_index(name='monthly_total')
)

In [13]:
# Eligible customers = at least one month with ≥50 bags
eligible_customers = (
    monthly_sum[monthly_sum['monthly_total'] >= 50][mobile_col]
    .unique()
)

In [14]:
frequency_df = (
    monthly_sum[monthly_sum['monthly_total'] >= 50]
    .groupby(mobile_col)['purchase_month']
    .nunique()
    .reset_index(name='frequency')
)


In [15]:
rfm_base = df[df[mobile_col].isin(eligible_customers)].groupby(mobile_col).agg(
    recency=(date_col, lambda x: (analysis_date - x.max()).days),
    monetary=(qty_col, 'sum')
).reset_index()


In [16]:
rfm_base.head()

,Mobile Number,recency,monetary
0,6006022935,2,3922
1,6114101702,7,1602
2,6200047973,32,2619
3,6200090275,2,2840
4,6200104263,4,1565


In [17]:
rfm = rfm_base.merge(frequency_df, on=mobile_col, how='left')
rfm['frequency'] = rfm['frequency'].fillna(0).astype(int)

In [18]:
rfm.head()

,Mobile Number,recency,monetary,frequency
0,6006022935,2,3922,12
1,6114101702,7,1602,8
2,6200047973,32,2619,9
3,6200090275,2,2840,9
4,6200104263,4,1565,12


In [19]:
def r_score(recency):
    if recency <= 30:
        return 4
    elif recency <= 60:
        return 3
    elif recency <= 90:
        return 2
    else:
        return 1

rfm['r_score'] = rfm['recency'].apply(r_score)


In [20]:
def f_score(freq):
    if freq >= 11:
        return 4
    elif freq >= 8:
        return 3
    elif freq >= 5:
        return 2
    else:
        return 1

rfm['f_score'] = rfm['frequency'].apply(f_score)

In [21]:
def m_score(monetary_bags):
    if monetary_bags >= 10 * 20 * 12:        # 2400 bags
        return 4
    elif monetary_bags >= 8 * 20 * 12:       # 1920 bags
        return 3
    elif monetary_bags >= 5 * 20 * 12:       # 1200 bags
        return 2
    else:
        return 1

rfm['m_score'] = rfm['monetary'].apply(m_score)

In [22]:
rfm['rfm_sum'] = (
    rfm['r_score'] * 0.15 +
    rfm['f_score'] * 0.60 +
    rfm['m_score'] * 0.25
)

In [23]:
#5. Rule-based Segmentation
# ==============================
def classify(total):
    if total == 4:
        return 'Loyal'
    elif total >= 3:
        return 'Stable'
    elif total >= 2:
        return 'Opportunist'
    else:
        return 'At Risk'

rfm['segment_rule'] = rfm['rfm_sum'].apply(classify)

In [24]:
last_info = (
    df.sort_values(date_col)
    .groupby(mobile_col)
    .last()
    .reset_index()[[mobile_col, region_col, branch_col, territory_col]]
    .rename(columns={
        region_col: 'region',
        branch_col: 'branch',
        territory_col: 'territory'
    })
)

rfm_final = rfm.merge(
    last_info,
    on=mobile_col,
    how='left'
)

rfm_final.rename(columns={mobile_col: 'customer_mobile'}, inplace=True)


In [25]:
# ==============================
# 8. Save & Output
# ==============================
rfm_final.to_excel(OUTPUT_PATH, index=False)
print(f"✅ RFM model completed and saved to: {OUTPUT_PATH}")

✅ RFM model completed and saved to: D:\Champion Club RFM Data_2.xlsx


In [26]:
# Summary of segment counts
print("\nRule-based segments:")
print(rfm_final['segment_rule'].value_counts())


Rule-based segments:
segment_rule
Loyal          5366
Stable         3701
Opportunist    2505
At Risk         414
Name: count, dtype: int64
